In [2]:
from CosmoSynth.models.parameters import Parameter
from typing import List, Tuple, Union, Callable, Dict

def test(a,b=2):
    return a + b

#### NUOVA CLASSE PER IL PARAMETER HANDLER

In [34]:
from typing import Iterator
from collections import OrderedDict
import numpy as np
# Assuming 'Parameter' is defined elsewhere
# from parameter_module import Parameter


class ParameterHandler:
    """
    Classe che gestisce un insieme di parametri per un modello.

    Attributes:
        _parameters (OrderedDict[str, Parameter]): Dizionario dei parametri.
        _is_inside_model (bool): Indica se il gestore è stato aggiunto a un modello.
        _cache (Dict[str, Any]): Cache per le proprietà dei parametri.
    """

    def __init__(
        self, parameters: Union["Parameter", List["Parameter"]] = None
    ) -> None:
        """
        Inizializza il gestore dei parametri.

        Args:
            parameters (Union[Parameter, List[Parameter]], opzionale): Un singolo parametro o una lista di parametri.
        """
        self._parameters = OrderedDict()
        self._is_inside_model = (
            False  # Una volta dentro modello non posso aggiungere parametri
        )
        self._cache = {}
        if isinstance(parameters, Parameter):
            self.add_parameter(parameters)
        elif isinstance(parameters, list):
            for param in parameters:
                self.add_parameter(param)
        elif parameters is None:
            pass
        else:
            raise TypeError(
                "Parameters must be of type Parameter or List[Parameter]",
                type(parameters),
            )

    def _invalidate_cache(self) -> None:
        """
        Invalida la cache dei parametri.
        """
        self._cache = {}

    @property
    def is_inside_model(self) -> bool:
        """
        Indica se il gestore è stato aggiunto a un modello.

        Returns:
            bool: True se è stato aggiunto a un modello, False altrimenti.
        """
        return self._is_inside_model

    def lock(self) -> None:
        """
        Blocca il gestore per prevenire l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = True

    def unlock(self) -> None:
        """
        Sblocca il gestore per permettere l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = False

    @property
    def parameters_values(self) -> List[float]:
        """
        Ritorna i valori dei parametri.

        Returns:
            List[float]: Lista dei valori dei parametri.
        """
        return [p.value for p in self]

    @property
    def parameters_names(self) -> List[str]:
        """
        Ritorna i nomi dei parametri.

        Returns:
            List[str]: Lista dei nomi dei parametri.
        """
        if "parameters_names" not in self._cache:
            self._cache["parameters_names"] = list(self._parameters.keys())
        return self._cache["parameters_names"]

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:
        """
        Ritorna i limiti dei parametri.

        Returns:
            List[Tuple[float, float]]: Lista dei limiti dei parametri.
        """
        return [p.bounds for p in self]

    @property
    def n_free_params(self) -> int:
        """
        Ritorna il numero di parametri liberi.

        Returns:
            int: Numero di parametri liberi.
        """
        return len(self.free_parameters)

    @property
    def free_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri liberi.

        Returns:
            List[Parameter]: Lista dei parametri liberi.
        """
        if "free_parameters" not in self._cache:
            self._cache["free_parameters"] = [p for p in self if not p.frozen]
        return self._cache["free_parameters"]

    @property
    def frozen_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri congelati.

        Returns:
            List[Parameter]: Lista dei parametri congelati.
        """
        if "frozen_parameters" not in self._cache:
            self._cache["frozen_parameters"] = [p for p in self if p.frozen]
        return self._cache["frozen_parameters"]

    def set_values(
        self, values: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i valori dei parametri.

        Args:
            values (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(values, "value", include_frozen=include_frozen)
        self._invalidate_cache()

    def set_bounds(
        self, bounds: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i limiti dei parametri.

        Args:
            bounds (Union[List, Dict]): Limiti da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(bounds, "bounds", include_frozen=include_frozen)
        self._invalidate_cache()

    def set_frozen(
        self, is_frozen: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta lo stato di congelamento dei parametri.

        Args:
            is_frozen (Union[List, Dict]): Stato di congelamento da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri già congelati. Default è False.
        """
        self._assign_attribute(is_frozen, "frozen", include_frozen=include_frozen)
        self._invalidate_cache()

    def _assign_attribute(
        self, items: Union[List, Dict], attribute: str, include_frozen: bool = False
    ) -> None:
        """
        Assegna un valore a un attributo dei parametri.

        Args:
            items (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            attribute (str): Nome dell'attributo da assegnare.
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.

        Raises:
            ValueError: Se il numero di elementi non corrisponde al numero di parametri.
            TypeError: Se items non è né una lista né un dizionario.
        """
        if isinstance(items, (list, np.ndarray)):
            params = list(self) if include_frozen else self.free_parameters
            if len(items) != len(params):
                raise ValueError(
                    f"Number of items {len(items)} must match number of {'all' if include_frozen else 'free'} parameters ({len(params)})"
                )
            for param, val in zip(params, items):
                setattr(param, attribute, val)
        elif isinstance(items, dict):
            for name, val in items.items():
                param = self[name]
                if not include_frozen and param.frozen:
                    continue
                setattr(param, attribute, val)
        else:
            raise TypeError("Items must be a list or dictionary")

    def add_parameter(self, parameter: "Parameter") -> None:
        """
        Aggiunge un parametro al gestore.

        Args:
            parameter (Parameter): Il parametro da aggiungere.

        Raises:
            ValueError: Se il parametro esiste già o se si tenta di aggiungere un parametro dopo la creazione del modello.
        """
        if self._is_inside_model:
            raise ValueError(
                f"Cannot add parameter {parameter.name} to model after it is locked"
            )
        if not isinstance(parameter, Parameter):
            raise TypeError('Addes parameter must be istance of Parameter class')
        if parameter.name in self._parameters:
            raise ValueError(f"Parameter {parameter.name} already exists.")
        self._parameters[parameter.name] = parameter
        self._invalidate_cache()

    def __getitem__(self, name: str) -> "Parameter":
        """
        Ritorna un parametro usando il suo nome.

        Args:
            name (str): Nome del parametro.

        Returns:
            Parameter: Il parametro richiesto.

        Raises:
            KeyError: Se il parametro non è trovato.
        """
        try:
            return self._parameters[name]
        except KeyError:
            raise KeyError(f"Parameter '{name}' not found.")

    def __setitem__(self, key: str, value: "Parameter") -> None:
        """
        Imposta un parametro usando il suo nome.

        Args:
            key (str): Nome del parametro.
            value (Parameter): Il parametro da impostare.

        Raises:
            ValueError: Se si tenta di impostare un parametro dopo la creazione del modello.
            TypeError: Se value non è un'istanza di Parameter.
            ValueError: Se il parametro esiste già.
        """
        if self._is_inside_model:
            raise ValueError(f"Cannot add parameter {key} to model after it is locked")
        if not isinstance(value, Parameter):
            raise TypeError(
                f"New parameter must be instance of Parameter, not {type(value)}"
            )
        if key in self._parameters:
            raise ValueError(f"Parameter {key} already exists.")
        self._parameters[key] = value
        self._invalidate_cache()

    def __contains__(self, key: str) -> bool:
        """
        Verifica se un parametro è presente usando il suo nome.

        Args:
            key (str): Nome del parametro.

        Returns:
            bool: True se il parametro è presente, False altrimenti.
        """
        return key in self._parameters

    def __iter__(self) -> Iterator["Parameter"]:
        """
        Itera sui parametri.

        Returns:
            Iterator[Parameter]: Iteratore sui parametri.
        """
        return iter(self._parameters.values())

    def __len__(self) -> int:
        """
        Ritorna il numero di parametri.

        Returns:
            int: Numero di parametri.
        """
        return len(self._parameters)
    
    def __str__(self) -> str:
        """
        Ritorna una rappresentazione in formato tabella dei parametri gestiti.

        Returns:
            str: Tabella che mostra nome, valore, bounds e stato frozen dei parametri.
        """
        # Creiamo l'header della tabella
        header = f"{'Name':<20} {'Value':<20} {'Bounds':<30} {'Frozen':<10}\n"
        header += "-" * 80 + "\n"
        
        # Raccogliamo i dati di ogni parametro
        rows = []
        for param in self:
            name = param.name
            value = str(param.value) if param.value is not None else "None"
            bounds = str(param.bounds) if param.bounds is not None else "None"
            frozen = "Yes" if param.frozen else "No"
            
            # Aggiungiamo una riga alla tabella
            row = f"{name:<20} {value:<20} {bounds:<30} {frozen:<10}"
            rows.append(row)
        
        # Uniamo l'header con tutte le righe
        table = header + "\n".join(rows)
        return table

    def items(self):
        """
        Ritorna gli elementi del gestore come coppie chiave-valore.

        Returns:
            ItemsView: Vista degli elementi del gestore.
        """
        return self._parameters.items()

    def keys(self):
        """
        Ritorna le chiavi del dizionario base.

        Returns:
            KeysView: Vista delle chiavi del gestore.
        """
        return self._parameters.keys()

    def values(self):
        """
        Ritorna i parametri.

        Returns:
            ValuesView: Vista dei valori del gestore.
        """
        return self._parameters.values()


# Esempi di utilizzo
p1 = Parameter(name="param1", value=5.0, bounds=(0.0, 10.0), frozen=True)
p2 = Parameter(name="param2", value=7.0, bounds=(0.0, 10.0))

handler = ParameterHandler([p1, p2])

print(handler.parameters_names)  # ['param1', 'param2']
print(handler.parameters_values)  # [5.0, 7.0]
print(handler.n_free_params)  # 1
print(
    handler.free_parameters
)  # [<Parameter name='param2', value=7.0, bounds=(0.0, 10.0), frozen=False>]
print(
    handler.frozen_parameters
)  # [<Parameter name='param1', value=5.0, bounds=(0.0, 10.0), frozen=True>]

handler.set_values({"param2": 8.0})
print(handler.parameters_values)  # [5.0, 8.0]

handler.set_bounds({"param2": (0.0, 15.0)})
print(handler.parameters_bounds)  # [(0.0, 10.0), (0.0, 15.0)]

handler.set_frozen({"param1": True})
print(handler.frozen_parameters)  # [<Parameter name='param1', val

print('param1' in handler, len(handler))

handler.set_frozen(is_frozen=[False, False], include_frozen=True)
handler['param1'].value = 2.55  #metodo migliore
print(handler.free_parameters)
print(handler['param1'])
print(handler['param2'])
print(handler)

['param1', 'param2']
[5.0, 7.0]
1
[5.0, 8.0]
[(0.0, 10.0), (0.0, 15.0)]
True 2
[<CosmoSynth.models.parameters.Parameter object at 0x7fcf2063d250>, <CosmoSynth.models.parameters.Parameter object at 0x7fcf2060e090>]
PARAM NAME: param1
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
param1          2.55       0          (0, 10)                                   

PARAM NAME: param2
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
param2          8          0          (0, 15)                                   

Name              

In [87]:
def gaussian(x, mu=0, sigma=1):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def calcola_dimensioni(lista):
    # Variabili di controllo per la sequenza
    dimensioni = 0
    if lista[0] == "x" and lista[1] != "y":
        dimensioni += 1
    elif lista[0] == "x" and lista[1] == "y" and lista[2] != "z":
        dimensioni += 2
    elif lista[0] == "x" and lista[1] == "y" and lista[2] == "z":
        dimensioni += 3

    return dimensioni


In [109]:
import inspect
class Model:
    def __init__(
        self, func, parameters, ndim, ninputs, noutputs, name="SimpleModel"
    ) -> None:
        
        self._parameters = parameters
        self._ndim = ndim
        self._ninputs = ninputs
        self._noutputs = noutputs
        self._callable = func
        self._name = name
        self._grid_variables = None  # Inizializza _grid_variables nel costruttore
        
        self._update_callable()

    # POINTER PROPRIETA
    @property
    def parameters(self):
        return self._parameters

    @property
    def name(self):
        return self._name

    @property
    def n_dim(self):
        return self._ndim

    @property
    def n_inputs(self):
        return self._ninputs

    @property
    def n_outputs(self):
        return self._noutputs

    @property
    def grid_variables(self):
        return self._grid_variables
    
    @property
    def parameters_names(self) -> List[str]:
        return self._parameters.parameters_names

    @property
    def n_parameters(self) -> int:
        return len(self.parameters)

    @property
    def parameters_values(self) -> List[float]:
        return self.parameters.parameters_values

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:        
        return self.parameters.parameters_bounds

    @property
    def free_parameters(self) -> List[Parameter]:       
        return self.parameters.free_parameters

    @property
    def frozen_parameters(self) -> List[Parameter]:
        return self.parameters.frozen_parameters

    @property
    def n_free_parameters(self) -> int:
        return self.parameters.n_free_params

    @property
    def _binary_freeze_map(self) -> List[bool]:
        return [p.frozen for p in self]

    @property
    def _binary_melt_map(self) -> List[bool]:
        return [not p.frozen for p in self]

    @staticmethod
    def _extract_params(method):
        

        """
        Estrae i nomi e i valori di default dei parametri dal metodo evaluate.

        Parameters:
        -----------
        method : function
            Metodo evaluate della classe.

        Returns:
        --------
        tuple
            Lista dei nomi dei parametri, dei valori di default e dello stato frozen.
        """
        signature = inspect.signature(method)
        params = {}
        is_constant = []
        for param_name, param in signature.parameters.items():
            if param_name != "self":
                if param.default is inspect.Parameter.empty:
                    params[param_name] = 1
                    is_constant.append(False)
                else:
                    params[param_name] = param.default
                    is_constant.append(True)
        return list(params.keys()), list(params.values()), is_constant

    @classmethod
    def from_callable(cls, func, ndim=None, ninputs=1, noutputs=1, name="SimpleModel"):
        names, values, frozen = cls._extract_params(func)

        # check sul numero di dimensioni
        if ndim is None:
            ndim = calcola_dimensioni(names)

        parameters = ParameterHandler()

        for i in range(ndim, len(names)):
            parameters.add_parameter(
                Parameter(name=names[i], value=values[i], frozen=frozen[i])
            )

        new_cls = cls(func, parameters, ndim, ninputs, noutputs, name)
        new_cls._grid_variables = names[:ndim]  # Sovrascrive _grid_variables
        new_cls._update_callable()
        return new_cls
    
    

    def __str__(self):
        """
        Restituisce una rappresentazione testuale del modello.

        Returns:
            str: Una stringa che rappresenta il modello.
        """
        total_string = f"MODEL NAME: {self.name} \n"
        total_string += f"FREE PARAMS: {self.n_free_parameters}\n"
        total_string += f"GRID VARIABLES: {self.grid_variables}\n"
        total_string += "-" * 60 + "\n"
        total_string += (
            f"{'':<4} {'NAME':<15} {'VALUE':<10} {'IS-FROZEN':<10} {'BOUNDS':<20}\n"
        )
        total_string += "-" * 60 + "\n"
        for i, param in enumerate(self._parameters):
            value_str = f"{param.value:.2f}"
            bounds_str = f"({param.bounds[0]:.2f}, {param.bounds[1]:.2f})"
            frz = "Yes" if param.frozen else "No"
            total_string += (
                f"{i:<4} {param.name:<15} {value_str:<10} {frz:<10} {bounds_str:<20}\n"
            )
        return total_string
    
    def _update_callable(self):
        # Ottiene i parametri liberi
        free_params = self.parameters.free_parameters
        free_param_names = [param.name for param in free_params]

        # Ottiene le variabili di griglia
        grid_variable_names = self._grid_variables if self._grid_variables else []

        # Crea una nuova funzione con la firma aggiornata
        def internal_callable(*args, **kwargs):
            
            bound_args = {}
            all_param_names = grid_variable_names + free_param_names
            bound_args.update(zip(all_param_names, args))
            bound_args.update(kwargs)

            # Include i parametri congelati
            for param in self.parameters.frozen_parameters:
                bound_args[param.name] = param.value

            # Chiama la funzione originale
            return self._callable(**bound_args)

        # Aggiorna la firma di internal_callable
        params = []
        for name in grid_variable_names:
            params.append(
                inspect.Parameter(
                    name,
                    inspect.Parameter.POSITIONAL_OR_KEYWORD,
                    default=inspect.Parameter.empty,
                )
            )
        for name in free_param_names:
            params.append(
                inspect.Parameter(
                    name,
                    inspect.Parameter.POSITIONAL_OR_KEYWORD,
                    default=inspect.Parameter.empty,
                )
            )

        internal_callable.__signature__ = inspect.Signature(params)

        # Imposta _internal_callable come la funzione interna
        self._internal_callable = internal_callable

    def __call__(self, *args, **kwargs):
        return self._internal_callable(*args, **kwargs)
    
    
    
mod = Model.from_callable(gaussian)
print(mod)
print(mod(3))

MODEL NAME: SimpleModel 
FREE PARAMS: 0
GRID VARIABLES: ['x']
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    mu              0.00       Yes        (-inf, inf)         
1    sigma           1.00       Yes        (-inf, inf)         

0.0044318484119380075


In [106]:
def multi_dimensional_model(x, y, z, w, param1, param2=3.5):
    # Implementazione del modello
    return x + y + z + w + param1 * param2


# Crea il modello utilizzando from_callable
mod = Model.from_callable(multi_dimensional_model)

print(mod)
# Chiama il modello con valori per le variabili di griglia
result = mod(1.0, 2, 4, w = 5, param1 = 2, param2 = 3)

print(result)


MODEL NAME: SimpleModel 
FREE PARAMS: 2
GRID VARIABLES: ['x', 'y', 'z']
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    w               1.00       No         (-inf, inf)         
1    param1          1.00       No         (-inf, inf)         
2    param2          3.50       Yes        (-inf, inf)         

19.0
